# PWV07c : Per night PWV curves an quality metrics For differrent Filters and including Selection cuts

**Goal :** PWV vs time in Spectrogram and Spectrum, and difference per observation

- author Sylvie Dagoret-Campagne
- creation date 2026-03-21 : version run2026_v02b_cr_run2026_v02d_cr
- last update : 2026-03-22
- affiliation : IJCLab
- Kernel @usdf **w_2026_02*
- Home emac : base (conda)
- laptop : conda_py313

**Goal** : Show PWV and diffPWV vs time and histograms and compare without and with cuts

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from platform import python_version
print(python_version())

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
# must install the mysitcom package by doing at top level "pip install --user -e . "
from mysitcom.auxtel.pwv import scatter_datetime
from mysitcom.auxtel.pwv import strip_datetime
from mysitcom.auxtel.pwv import bar_counts_by_night
from mysitcom.auxtel.pwv import plot_dccd_chi2_vs_time
from mysitcom.auxtel.pwv import plot_dccd_chi2_vs_time_by_filter
from mysitcom.auxtel.pwv import stripplot_target_vs_time
from mysitcom.auxtel.pwv import plot_dccd_chi2_vs_time_by_target_filter
from mysitcom.auxtel.pwv import plot_dccd_chi2_histo_by_target_filter
from mysitcom.auxtel.pwv import plot_dccd_chi2_vs_time_by_target_filter_colorsedtype
from mysitcom.auxtel.pwv import plot_dccd_chi2_histo_by_target_filter_colorsedtype
from mysitcom.auxtel.pwv import summarize_dccd_chi2
from mysitcom.auxtel.pwv import plot_atmparam_vs_time, plot_atmparam_diff_vs_time
from mysitcom.auxtel.pwv import plot_atmparam_hist_per_filter, plot_atmparam_diff_hist_per_filter
from mysitcom.auxtel.pwv import plot_atmparam_vs_time_byfilter_bytarget,plot_atmparam_hist_stacked_bytarget
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter
from mysitcom.auxtel.pwv import GetNightMidnightsDict,GetNightBoundariesDict

In [ ]:
from mysitcom.auxtel.pwv import pwv_deviation_from_linear_interp_datetime
from mysitcom.auxtel.pwv import pwv_next_prev_difference_datetime
from mysitcom.auxtel.pwv import plot_atmparam_hist_per_filter
from mysitcom.auxtel.pwv import compute_atmparam_stats_per_filter
from mysitcom.auxtel.pwv import plot_atm2param_vs_time

In [ ]:
from mysitcom.auxtel.qualitycuts import ParameterCutSelection,ParameterCutTools,ParameterCutPlotting

In [ ]:
import os

In [ ]:
# where are stored the figures
pathfigs = "figs_PWV07c"
prefix = "pwv07c"
if not os.path.exists(pathfigs):
    os.makedirs(pathfigs) 
figtype = ".png"

In [ ]:
pathdata = "data_PWV07c"
if not os.path.exists(pathdata):
    os.makedirs(pathdata) 

In [ ]:
import numpy as np
from numpy.linalg import inv
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%matplotlib inline
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm,SymLogNorm
from matplotlib.patches import Circle,Annulus
from astropy.visualization import ZScaleInterval
props = dict(boxstyle='round', facecolor="white", alpha=0.1)
#props = dict(boxstyle='round')

import matplotlib.colors as colors
import matplotlib.cm as cmx

import matplotlib.ticker                         # here's where the formatter is
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

from matplotlib.gridspec import GridSpec


from matplotlib.lines import Line2D
from matplotlib.dates import DateFormatter



from astropy.visualization import (MinMaxInterval, SqrtStretch,ZScaleInterval,PercentileInterval,
                                   ImageNormalize,imshow_norm)
from astropy.visualization.stretch import SinhStretch, LinearStretch,AsinhStretch,LogStretch

from astropy.io import fits
from astropy.wcs import WCS
from astropy import units as u
from astropy import constants as c

from scipy import interpolate
from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import KDTree, BallTree

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option('display.max_rows', 100)

import matplotlib.ticker                         # here's where the formatter is
import os
import re
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype

import pickle
from collections import OrderedDict

plt.rcParams["figure.figsize"] = (16,8)
plt.rcParams["axes.labelsize"] = 'xx-large'
plt.rcParams['axes.titlesize'] = 'xx-large'
plt.rcParams['xtick.labelsize']= 'xx-large'
plt.rcParams['ytick.labelsize']= 'xx-large'
#plt.rcParams["legend.fontsize"] = "xx-large"

import scipy
from scipy.optimize import curve_fit,least_squares

from pprint import pprint

# new color correction model
import pickle
from scipy.interpolate import RegularGridInterpolator

In [ ]:
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

from astropy.visualization import (MinMaxInterval, SqrtStretch,ZScaleInterval,PercentileInterval,
                                   ImageNormalize,imshow_norm)
from astropy.visualization.stretch import SinhStretch, LinearStretch,AsinhStretch,LogStretch

from astropy.time import Time


In [ ]:
from getCalspec import getCalspec
from getCalspec.getCalspec import getCalspecDataFrame

In [ ]:
# Remove to run faster the notebook
#import ipywidgets as widgets
#%matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
#try:
#    import ipympl  # noqa: F401
#    %matplotlib widget
#    print("ipympl found → interactive backend (%matplotlib widget)")
#except ImportError:
#    %matplotlib inline
#    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
#    print("Install with:  pip install ipympl")

In [ ]:
from PWV00_parameters import *

In [ ]:
DumpConfig()

In [ ]:
from importlib.metadata import version

In [ ]:
# wavelength bin colors
#jet = plt.get_cmap('jet')
#cNorm = mpl.colors.Normalize(vmin=0, vmax=NSED)
#scalarMap = cmx.ScalarMappable(norm=cNorm, cmap=jet)
#all_colors = scalarMap.to_rgba(np.arange(NSED), alpha=1)

In [ ]:
np.__version__

In [ ]:
pd.__version__

### Configuration

In [ ]:
def convertNumToDatestr(num):
    year = num//10_000
    month= (num-year*10_000)//100
    day = (num-year*10_000-month*100)

    year_str = str(year).zfill(4)
    month_str = str(month).zfill(2)
    day_str = str(day).zfill(2)
    
    datestr = f"{year_str}-{month_str}-{day_str}"
    return pd.to_datetime(datestr)

In [ ]:
PWVMIN = 0.
PWVMAX = 10.

In [ ]:
FLAG_WITHCOLLIMATOR = True
DATE_WITHCOLLIMATOR = 20230930
datetime_WITHCOLLIMATOR = convertNumToDatestr(DATE_WITHCOLLIMATOR)
datetime_WITHCOLLIMATOR = pd.to_datetime("2023-09-30 00:00:00.0+0000")
datetime_WITHCOLLIMATOR

## Type of quality cut

In [ ]:
FLAG_LOOSE_CUTS = False
FLAG_TIGHT_CUTS = True

## Flag Vertical lines for nights

In [ ]:
FLAG_NIGHT_VERTICALLINES = True

## Initialisation

### Read the file
- `atmfilename` is defined in `PWV00_parameters.py` 

In [ ]:
# apply if using Corentin's concatenated files
if version_run  in ["run2026_v02d_cr", "run2026_v02b_cr_run2026_v02c_cr", "run2026_v02b_cr_run2026_v02d_cr"]:
    atmfilename = extractedfilesdict[version_run] 
print(f"INPUT FILENAME{atmfilename}")    

In [ ]:
the_suptitle = butlerusercollectiondict[version_run] 

In [ ]:
import sys
import numpy as np
import types
sys.modules['numpy.rec'] = np.rec

In [ ]:
inputfilename = atmfilename.split("/")[-1]

if "parquet" in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
elif "npy" in inputfilename and "run_v" in version_run :
    specdata = np.load(atmfilename,allow_pickle=True)
    df_spec = pd.DataFrame(specdata)
    df_spec["D_CCD [mm]"] = df_spec["D2CCD"]
    df_spec["PWV [mm]"] = df_spec["PWV [mm]_x"] 
    df_spec["PWV [mm]_rum"] = df_spec["PWV [mm]_y"] 
    df_spec["PWV [mm]_err"] = df_spec["PWV [mm]_err_x"] 
    df_spec["PWV [mm]_err_rum"] = df_spec["PWV [mm]_err_y"] 


    cols = [
    "PWV [mm]",
    "PWV [mm]_rum",
    "PWV [mm]_err",
    "PWV [mm]_err_rum",
    ]
    df_spec = df_spec.dropna(subset=cols)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec = pd.DataFrame(specdata)
    
#else:
#    raise "bad path of filename {inputfilename}"
    

In [ ]:
FLAG_RENAME_SPECTROGRAM_VARIABLES = True

if FLAG_RENAME_SPECTROGRAM_VARIABLES and "run2026_v01" in version_run:
    df_spec.rename(
    {
    "chi2":"chi2_ram",
    "A1":"A1_ram",
    "A1_err": "A1_err_ram",
    "A2": "A2_ram",
    "A2_err": "A2_err_ram",
    "A3": "A3_ram",
    "A3_err": "A3_err_ram", 
    "VAOD": "VAOD_ram", 
    "VAOD_err": "VAOD_err_ram", 
    "angstrom_exp" : "angstrom_exp_ram", 
    "angstrom_exp_err" : "angstrom_exp_err_ram" , 
    "ozone [db]" :"ozone [db]_ram", 
    "ozone [db]_err": "ozone [db]_err_ram", 
    "PWV [mm]":  "PWV [mm]_ram",
    "PWV [mm]_err":"PWV [mm]_err_ram" , 
    "B": "B_ram" , 
    "B_err" : "B_err_ram", 
    "A_star": "A_star_ram" , 
    "A_star_err": "A_star_err_ram" , 
    "D_CCD [mm]" : "D_CCD [mm]_ram", 
    "D_CCD [mm]_err": "D_CCD [mm]_err_ram" 
    }
    ,axis=1,inplace = True)
elif FLAG_RENAME_SPECTROGRAM_VARIABLES and "run2026_v02" in version_run:
    df_spec["chi2_ram"] = df_spec["CHI2_FIT"]
    df_spec["chi2_rum"] = df_spec["CHI2_FIT"]

    

In [ ]:
print(" | ".join(df_spec.columns)) 

In [ ]:
#df_spec.dtypes.to_frame('Type de donnée')

In [ ]:
# 1. Convert to datetime
#df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"])
df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"],utc=True)
# 2. Sort by time (chronological order)
df_spec = df_spec.sort_values(by="Time", ascending=True).reset_index(drop=True)

In [ ]:
if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec["Time"] > datetime_WITHCOLLIMATOR].copy() 

In [ ]:
df_spec["nightObs"] = df_spec.apply(lambda x: x['id']//100_000, axis=1)

In [ ]:
df_spec["seq_num"]  = df_spec["id"] % 100_000

In [ ]:
df_spec["night_from_time_utc"] = (
    df_spec["Time"].dt.strftime("%Y%m%d").astype(int)
)

mismatch = df_spec[
    df_spec["night_from_time_utc"] != df_spec["nightObs"]
]

len(mismatch), len(df_spec)

In [ ]:
print(f"*** VERSION_RUN : {version_run} ***")

In [ ]:
FLAG_CORRECT_NIGHTOBS_VARIABLES = True

if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    print("COMPUTE NIGHTOBS FROM Time") 
    # Assurer UTC
    #df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"], utc=True)

    # Convertir au Chili
    df_spec["Time_local"] = df_spec["Time"].dt.tz_convert("America/Santiago")

    # Décalage de 12h vers le passé
    df_spec["nightObs_corr"] = (
        (df_spec["Time_local"] - pd.Timedelta(hours=12))
        .dt.strftime("%Y%m%d")
        .astype(int)
    )

In [ ]:
if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    display(df_spec[["id","DATE-OBS","nightObs","nightObs_corr"]])

In [ ]:
if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    span = (
        df_spec.groupby("nightObs_corr")["Time"].max()
        - df_spec.groupby("nightObs_corr")["Time"].min()
    )

    print(span.max())

In [ ]:
if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    df_spec["nightObs"] = df_spec["nightObs_corr"]
    

## Select only empty and OG550 filters

In [ ]:
df_spec["FILTER"].unique()

In [ ]:
if FLAG_PWVFILTERS: 
    df_spec = df_spec[df_spec["FILTER"].isin(PWV_FILTER_LIST) ]

### Check Filters

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 4))

strip_datetime(
    df=df_spec,
    x="Time",
    y="FILTER",
    hue="FILTER",
    ax=ax,
    size=9,
)

ax.set_title(tag)
plt.show()


## Palette with more clear target seperation colors

In [ ]:
# Comptage et tri
target_counts = (
    df_spec['TARGET']
    .value_counts()
    .sort_values(ascending=False)
)
targets = target_counts.index.tolist()
counts = target_counts.values

In [ ]:
chosen_palette = "tab20"

if chosen_palette == "husl":
    palette = sns.color_palette("husl", n_colors=len(targets))
    #palette = sns.color_palette("husl", len(targets))[::-1]
    target_color_map = OrderedDict(zip(targets, palette))
elif chosen_palette == "hsv":
    base_palette = sns.color_palette("hsv", n_colors=len(targets))
    # réordonnancement pour maximiser contraste local
    order = np.arange(len(base_palette))
    order = np.roll(order, len(order)//2)
    palette = [base_palette[i] for i in order]
    target_color_map = OrderedDict(zip(targets, palette))
    #target_color_map = OrderedDict(zip(targets, palette[::-1]))
elif chosen_palette == "tab20":
    palette = sns.color_palette("tab20b", 20) + sns.color_palette("tab20c", 10)
    palette = palette[:len(targets)]
    target_color_map = OrderedDict(zip(targets, palette))
    #target_color_map = OrderedDict(zip(targets, palette[::-1]))
else:
    palette = sns.color_palette("viridis", n_colors=len(targets))
    #palette = sns.color_palette("viridis", n_colors=len(targets))[::-1]
    target_color_map = OrderedDict(zip(targets, palette)) 

# Colormap discrete
cmap = mpl.colors.ListedColormap(palette)
norm = mpl.colors.BoundaryNorm(boundaries=range(len(targets)+1),ncolors=len(targets))

In [ ]:
ordered_list_of_targets = list(target_color_map.keys())

In [ ]:
fig = plt.figure(figsize=(0.8*len(targets), 3),layout="constrained")

# axe très épais (0.15)
cax = fig.add_axes([0.05, 0.7, 0.95, 0.15])  
# [left, bottom, width, height]

cb = mpl.colorbar.ColorbarBase(
    cax,
    cmap=cmap,
    norm=norm,
    orientation='horizontal'
)

cb.set_ticks([i + 0.5 for i in range(len(targets))])
cb.set_ticklabels(targets)
cb.ax.tick_params(labelrotation=90)
cb.set_label("TARGET", labelpad=10)
cb.ax.tick_params(labelsize=10,length=6,width=1.5)


plt.suptitle(tag)
figfilename = f"{pathfigs}/{prefix}_{version_run}_palette_{chosen_palette}_targetnames{figtype}"
fig.savefig(figfilename)

#fig.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 0.35*len(targets)),layout="constrained")

sns.barplot(
    x=counts,
    y=targets,
    palette=palette,
    ax=ax,
)

ax.set_xlabel("Number of Obs")
ax.set_ylabel("TARGET")
ax.set_title("TARGET observed")
ax.tick_params(axis="y", labelsize=10)
ax.grid()
#plt.tight_layout()

plt.suptitle(the_suptitle)
figfilename = f"{pathfigs}/{prefix}_{version_run}_baplottargets_palette_{chosen_palette}{figtype}"
plt.savefig(figfilename)
plt.show()

## Processing of some quantities

### calculate diff_PWV

In [ ]:
denom = np.sqrt(df_spec["PWV [mm]_err_ram"]**2 + df_spec["PWV [mm]_err_rum"]**2)

df_spec["diff_PWV_norm"] = np.where(
    np.isfinite(denom) & (denom > 0),
    (df_spec["PWV [mm]_ram"] - df_spec["PWV [mm]_rum"]) / denom,
    np.nan
)

df_spec["diff_PWV"] =  (df_spec["PWV [mm]_ram"] - df_spec["PWV [mm]_rum"]) 
df_spec["diff_PWV_err"] = np.sqrt( (df_spec["PWV [mm]_err_ram"]**2 - df_spec["PWV [mm]_err_rum"]**2)) 


### calculate chi2_norm

In [ ]:
df_spec, df1 = normalize_column_data_bytarget_byfilter(df_spec,target_col="TARGET",filter_col="FILTER",feature_col= "CHI2_FIT",ext="norm")
df_spec, df2 = normalize_column_data_bytarget_byfilter(df_spec,target_col="TARGET",filter_col="FILTER",feature_col= "chi2_ram",ext="norm")
df_spec, df3 = normalize_column_data_bytarget_byfilter(df_spec,target_col="TARGET",filter_col="FILTER",feature_col= "chi2_rum",ext="norm")

## Apply Quality cuts 

In [ ]:
if FLAG_LOOSE_CUTS:
    filename_cuts_final = f"{pathdata}/cuts_loose_finaldecision.json" 
    cutstype_name = "loose-cuts"
elif FLAG_TIGHT_CUTS: 
    filename_cuts_final = f"{pathdata}/cuts_tight_finaldecision.json" 
    cutstype_name = "tight-cuts"
else:
    filename_cuts_final = f"{pathdata}/cuts_finaldecision.json" 
    cutstype_name = "standard-cuts"

filename_cuts_short = os.path.basename(filename_cuts_final)


### Parameters cuts load

In [ ]:
cuts = ParameterCutTools.load_cuts_json(filename_cuts_final)

In [ ]:
#ParameterCutTools.cuts_to_dataframe(cuts)

In [ ]:
list_of_params = list(cuts.keys())
print(list_of_params)

### Filter 

In [ ]:
selector = ParameterCutSelection(
    df_spec,
    params = list_of_params ,
    id_col="id"
)

flags = selector.apply_cuts(cuts)
df_stats = selector.selection_statistics(cuts)
df_stats_v2 = selector.selection_statistics_inoutliers_by_param(cuts)

df_selected = df_spec.merge(flags, on="id")
df_keep = df_selected[df_selected["pass_all_cuts"]]

In [ ]:
N_init = len(df_spec)
N_keep = len(df_keep)
print(f"Selection results ({cutstype_name}) {N_keep} passed / {N_init}")

## Plot PWV vs time before and after cuts

## Calculate midnights and night boundaries

In [ ]:
DT = pd.Timedelta(minutes=7*24*60)
TMIN  = df_spec["Time"].min()-DT
TMAX  = df_spec["Time"].max()+DT

In [ ]:
# get night boundaries
dn = GetNightBoundariesDict(df_spec)
# get midnights
dnidnights = GetNightMidnightsDict(df_spec)

In [ ]:
df_spec["FILTER"].unique()

### Plot PWV in spectrogram vs Time before quality cuts and after

In [ ]:
fig,axs = plt.subplots(2,1,figsize=(18,10), layout="constrained",sharex=True)
ax1,ax2 = axs

#top plot without cuts
plot_atmparam_vs_time(
    df_spec,
    time_col= "Time",
    filter_col = "FILTER",
    param_col = "PWV [mm]_ram",
    param_err_col = "PWV [mm]_err_ram",
    title_param = "PWV vs time (spectrogram no cut qual. cut)",
  
    
    # seuils / bornes
    param_min_fig=PWVMIN,
    param_max_fig=PWVMAX,
    param_min_cut=None,
    param_max_cut=None,
 
    # titres
    #suptitle= suptitle,

    # axes externes
    axs=ax1
)



if FLAG_NIGHT_VERTICALLINES:
    if version_run not in ["run_v12"]:
        for key, tt in dn.items():
            ax1.axvspan(tt[0],tt[1], color='blue', alpha=0.05,lw=0.5)

    for key, midn in dnidnights.items():
        ax1.axvline( midn ,color="purple",ls=":",alpha=0.5,lw=0.5)
else:
    #ax1.grid()
    pass

#bottom plot with cuts
plot_atmparam_vs_time(
    df_keep,
    time_col= "Time",
    filter_col = "FILTER",
    param_col = "PWV [mm]_ram",
    param_err_col = "PWV [mm]_err_ram",
    title_param = f"PWV vs time (spectrogram with cuts :: file = {filename_cuts_short})",
    
    
    # seuils / bornes
    param_min_fig=PWVMIN,
    param_max_fig=PWVMAX,
    param_min_cut=None,
    param_max_cut=None,
 
    # titres
    #suptitle = suptitle

    # axes externes
    axs=ax2
)

if FLAG_NIGHT_VERTICALLINES:
    if version_run not in ["run_v12"]:
        for key, tt in dn.items():
            ax2.axvspan(tt[0],tt[1], color='blue', alpha=0.05,lw=0.5)

    for key, midn in dnidnights.items():
        ax2.axvline( midn ,color="purple",ls=":",alpha=0.5,lw=0.5)
else:
    #ax2.grid()
    pass


plt.suptitle(the_suptitle)
figname =f"{pathfigs}/{prefix}_{version_run}_plotvstime_pwvram_allfilters-{cutstype_name}"+figtype
plt.savefig(figname)
plt.show()

## Compute All Metrics

### Metrics Diff PWV

#### Metrics Diff PWV : No cut

In [ ]:
fig,ax = plot_atmparam_hist_per_filter(
    df_spec,
    filter_col="FILTER",
    param_col = "diff_PWV",
    param_range = (-1.,1.),

    # histogram control
    bins=100,
    density=True,
    hist_alpha=0.4,

    # x-axis limits
    param_min_fig=-1.,
    param_max_fig=1.,

    title_param="$\Delta$ PWV (diff) - No cut",
    # titres
    suptitle= the_suptitle
)

plt.suptitle(the_suptitle)
plt.tight_layout()
figname =f"{pathfigs}/{prefix}_{version_run}_plotplothistoliny_diffPWV_emptyog550filt_nocut"+figtype
plt.savefig(figname)
plt.show()

#### Metrics Diff PWV : With cut

In [ ]:
fig,ax = plot_atmparam_hist_per_filter(
    df_keep,
    filter_col="FILTER",
    param_col = "diff_PWV",
    param_range = (-1.,1.),

    # histogram control
    bins=100,
    density=True,
    hist_alpha=0.4,

    # x-axis limits
    param_min_fig=-1.,
    param_max_fig=1.,

    title_param="$\Delta$ PWV (diff)" + f"with cut {cutstype_name}",
    # titres
    #suptitle= the_suptitle
    figsize=(8,6)
)

plt.suptitle(the_suptitle)
plt.tight_layout()
figname =f"{pathfigs}/{prefix}_{version_run}_plotplothistoliny_diffPWV_emptyog550filt_cut{ cutstype_name}"+figtype
plt.savefig(figname)
plt.show()

In [ ]:
df_pwvdiff_nocut = compute_atmparam_stats_per_filter(
    df_spec,
    filter_col="FILTER",
    param_col = "diff_PWV")

In [ ]:
df_pwvdiff_nocut.style.set_caption(
    f"Table 1 — diffpwv - no cut") \
    .format("{:.3f}") \
    .set_table_styles(
    [{'selector': 'caption',
      'props': [('font-size', '16px'),
                ('font-weight', 'bold')]}]
)

In [ ]:
df_pwvdiff_cut = compute_atmparam_stats_per_filter(
    df_keep,
    filter_col="FILTER",
    param_col = "diff_PWV")

In [ ]:
df_pwvdiff_cut.style.set_caption(
    f"Table 2 — diffpwv - with cut {cutstype_name}") \
    .format("{:.3f}") \
    .set_table_styles(
    [{'selector': 'caption',
      'props': [('font-size', '16px'),
                ('font-weight', 'bold')]}]
    )

### Metrics PWV repeatability with time interpolation

#### Metrics PWV repeatability with time interpolation - No cut

In [ ]:
# Compute metrics on Spectrogram
df_spec = pwv_deviation_from_linear_interp_datetime(
    df_spec,
    night_col="nightObs",
    filter_col="FILTER",
    target_col="TARGET",
    time_col="Time",
    pwv_col="PWV [mm]_ram",
    suffix="repeatinterp_ram",
    time_unit="min",
)

In [ ]:
# compute metric on Spectrum
df_spec = pwv_deviation_from_linear_interp_datetime(
    df_spec,
    night_col="nightObs",
    filter_col="FILTER",
    target_col="TARGET",
    time_col="Time",
    pwv_col="PWV [mm]_rum",
    suffix="repeatinterp_rum",
    time_unit="min",
)

##### additional processing of repeatability after calculation

In [ ]:
df_spec["dt_repeatinterp_ram"] = pd.to_timedelta(df_spec["dt_repeatinterp_ram"])
df_spec["dt_repeatinterp_rum"] = pd.to_timedelta(df_spec["dt_repeatinterp_rum"])
df_spec["dt_repeatinterp_ram_min"] = df_spec["dt_repeatinterp_ram"].dt.total_seconds() / 60
df_spec["dt_repeatinterp_rum_min"] = df_spec["dt_repeatinterp_rum"].dt.total_seconds() / 60

In [ ]:
fig,ax = plot_atmparam_hist_per_filter(
    df_spec,
    filter_col="FILTER",
    param_col = "PWV [mm]_ram_repeatinterp_ram",
    param_range = (-1.,1.),

    # histogram control
    bins=100,
    density=True,
    hist_alpha=0.4,

    # x-axis limits
    param_min_fig=-1.,
    param_max_fig=1.,

    title_param="$\Delta$ PWV (interp repeatability in spectrogram) - No cut",
    # titres
    #suptitle= the_suptitle
    figsize=(8,6)
)
plt.suptitle(the_suptitle)
plt.tight_layout()
figname =f"{pathfigs}/{prefix}_{version_run}_plotplothistoliny_DPWVram_repeatinterp_emptyog550filt_nocut"+figtype
plt.savefig(figname)
plt.show()

#### Metrics PWV repeatability with time interpolation - With cut

In [ ]:
# Compute metrics on Spectrogram
df_keep = pwv_deviation_from_linear_interp_datetime(
    df_keep,
    night_col="nightObs",
    filter_col="FILTER",
    target_col="TARGET",
    time_col="Time",
    pwv_col="PWV [mm]_ram",
    suffix="repeatinterp_ram",
    time_unit="min",
)

In [ ]:
# compute metric on Spectrum
df_keep = pwv_deviation_from_linear_interp_datetime(
    df_keep,
    night_col="nightObs",
    filter_col="FILTER",
    target_col="TARGET",
    time_col="Time",
    pwv_col="PWV [mm]_rum",
    suffix="repeatinterp_rum",
    time_unit="min",
)

##### additional processing of repeatability after calculation

In [ ]:
df_keep["dt_repeatinterp_ram"] = pd.to_timedelta(df_keep["dt_repeatinterp_ram"])
df_keep["dt_repeatinterp_rum"] = pd.to_timedelta(df_keep["dt_repeatinterp_rum"])
df_keep["dt_repeatinterp_ram_min"] = df_keep["dt_repeatinterp_ram"].dt.total_seconds() / 60
df_keep["dt_repeatinterp_rum_min"] = df_keep["dt_repeatinterp_rum"].dt.total_seconds() / 60

In [ ]:
fig,ax = plot_atmparam_hist_per_filter(
    df_keep,
    filter_col="FILTER",
    param_col = "PWV [mm]_ram_repeatinterp_ram",
    param_range = (-1.,1.),

    # histogram control
    bins=100,
    density=True,
    hist_alpha=0.4,

    # x-axis limits
    param_min_fig=-1.,
    param_max_fig=1.,

    title_param="$\Delta$ PWV (interp repeatability in spectrogram) -" + f"with cut {cutstype_name}",
    # titres
    #suptitle= the_suptitle
    figsize=(8,6)
)
plt.suptitle(the_suptitle)
plt.tight_layout()
figname =f"{pathfigs}/{prefix}_{version_run}_plotplothistoliny_DPWVram_repeatinterp_emptyog550filt_cut{cutstype_name}"+figtype
plt.savefig(figname)
plt.show()

In [ ]:
df_pwv_repeatinterp_ram_nocut = compute_atmparam_stats_per_filter(
    df_spec,
    filter_col="FILTER",
    param_col = "PWV [mm]_ram_repeatinterp_ram")

In [ ]:
df_pwv_repeatinterp_ram_nocut.style.set_caption(
    f"Table 3 — pwv_repeatinterp_ram - no cut") \
    .format("{:.3f}") \
    .set_table_styles(
    [{'selector': 'caption',
      'props': [('font-size', '16px'),
                ('font-weight', 'bold')]}]
)

In [ ]:
df_pwv_repeatinterp_ram_cut = compute_atmparam_stats_per_filter(
    df_keep,
    filter_col="FILTER",
    param_col = "PWV [mm]_ram_repeatinterp_ram")

In [ ]:
df_pwv_repeatinterp_ram_cut.style.set_caption(
    f"Table 4 — pwv_repeatinterp_ram - cut {cutstype_name}") \
    .format("{:.3f}") \
    .set_table_styles(
    [{'selector': 'caption',
      'props': [('font-size', '16px'),
                ('font-weight', 'bold')]}]
)

### Metrics PWV repeatability with PWV difference next-previous

#### Metrics PWV repeatability with PWV difference - No cut

In [ ]:
# spectrogram
df_spec = pwv_next_prev_difference_datetime(
    df_spec,
    night_col="nightObs",
    filter_col="FILTER",
    target_col="TARGET",
    time_col="Time",
    pwv_col="PWV [mm]_ram",
    suffix="repeatdiff_ram",
    time_unit="min",
)

In [ ]:
# spectrum
df_spec = pwv_next_prev_difference_datetime(
    df_spec,
    night_col="nightObs",
    filter_col="FILTER",
    target_col="TARGET",
    time_col="Time",
    pwv_col="PWV [mm]_rum",
    suffix="repeatdiff_rum",
    time_unit="min",
)

##### additional processing of repeatability after calculation

In [ ]:
df_spec["dt_repeatdiff_ram"] = pd.to_timedelta(df_spec["dt_repeatdiff_ram"])
df_spec["dt_repeatdiff_rum"] = pd.to_timedelta(df_spec["dt_repeatdiff_rum"])
df_spec["dt_repeatdiff_ram_min"] = df_spec["dt_repeatdiff_ram"].dt.total_seconds() / 60
df_spec["dt_repeatdiff_rum_min"] = df_spec["dt_repeatdiff_rum"].dt.total_seconds() / 60

In [ ]:
fig,ax = plot_atmparam_hist_per_filter(
    df_spec,
    filter_col="FILTER",
    param_col = "PWV [mm]_ram_repeatdiff_ram",
    param_range = (-1.,1.),

    # histogram control
    bins=100,
    density=True,
    hist_alpha=0.4,

    # x-axis limits
    param_min_fig=-1.,
    param_max_fig=1.,

    title_param="$\Delta$ PWV (diff repeatability in spectrogram) - No cut",
    # titres
    #suptitle= the_suptitle
    figsize=(8,6)
)
plt.suptitle(the_suptitle)
plt.tight_layout()
figname =f"{pathfigs}/{prefix}_{version_run}_plotplothistoliny_DPWVramrepeatdiff_emptyog550filt_nocut"+figtype
plt.savefig(figname)
plt.show()

#### Metrics PWV repeatability with PWV difference - With cut

In [ ]:
# spectrogram
df_keep = pwv_next_prev_difference_datetime(
    df_keep,
    night_col="nightObs",
    filter_col="FILTER",
    target_col="TARGET",
    time_col="Time",
    pwv_col="PWV [mm]_ram",
    suffix="repeatdiff_ram",
    time_unit="min",
)

In [ ]:
# spectrum
df_keep = pwv_next_prev_difference_datetime(
    df_keep,
    night_col="nightObs",
    filter_col="FILTER",
    target_col="TARGET",
    time_col="Time",
    pwv_col="PWV [mm]_rum",
    suffix="repeatdiff_rum",
    time_unit="min",
)

##### additional processing of repeatability after calculation

In [ ]:
df_keep["dt_repeatdiff_ram"] = pd.to_timedelta(df_keep["dt_repeatdiff_ram"])
df_keep["dt_repeatdiff_rum"] = pd.to_timedelta(df_keep["dt_repeatdiff_rum"])
df_keep["dt_repeatdiff_ram_min"] = df_keep["dt_repeatdiff_ram"].dt.total_seconds() / 60
df_keep["dt_repeatdiff_rum_min"] = df_keep["dt_repeatdiff_rum"].dt.total_seconds() / 60

In [ ]:
fig,ax = plot_atmparam_hist_per_filter(
    df_keep,
    filter_col="FILTER",
    param_col = "PWV [mm]_ram_repeatdiff_ram",
    param_range = (-1.,1.),

    # histogram control
    bins=100,
    density=True,
    hist_alpha=0.4,

    # x-axis limits
    param_min_fig=-1.,
    param_max_fig=1.,

    title_param="$\Delta$ PWV (diff repeatability in spectrogram) -" + f"with cut {cutstype_name}",
    # titres
    #suptitle= the_suptitle
    figsize=(8,6)
)
plt.suptitle(the_suptitle)
plt.tight_layout()
figname =f"{pathfigs}/{prefix}_{version_run}_plotplothistoliny_DPWVramrepeatdiff_emptyog550filt_cut{cutstype_name}"+figtype
plt.savefig(figname)
plt.show()

In [ ]:
df_pwv_repeatdiff_ram_nocut = compute_atmparam_stats_per_filter(
    df_spec,
    filter_col="FILTER",
    param_col = "PWV [mm]_ram_repeatdiff_ram")

In [ ]:
df_pwv_repeatdiff_ram_nocut.style.set_caption(
    f"Table 5 — pwv_repeatdiff_ram - no cut") \
    .format("{:.3f}") \
    .set_table_styles(
    [{'selector': 'caption',
      'props': [('font-size', '16px'),
                ('font-weight', 'bold')]}]
)

In [ ]:
df_pwv_repeatdiff_ram_cut = compute_atmparam_stats_per_filter(
    df_keep,
    filter_col="FILTER",
    param_col = "PWV [mm]_ram_repeatdiff_ram")

In [ ]:
df_pwv_repeatdiff_ram_cut.style.set_caption(
    f"Table 5 — pwv_repeatdiff_ram - with cut {cutstype_name}") \
    .format("{:.3f}") \
    .set_table_styles(
    [{'selector': 'caption',
      'props': [('font-size', '16px'),
                ('font-weight', 'bold')]}]
)

In [ ]:
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(14,6))
df_keep["dt_repeatinterp_rum_min"].hist(bins=100,histtype="step",ax=ax1)
df_keep["dt_repeatinterp_ram_min"].hist(bins=100,histtype="step",ax=ax1)
df_keep["dt_repeatdiff_rum_min"].hist(bins=100,histtype="step",ax=ax2)
df_keep["dt_repeatdiff_ram_min"].hist(bins=100,histtype="step",ax=ax2)
ax1.set_xlabel("$\Delta t$ (min)")
ax2.set_xlabel("$\Delta t$ (min)")
plt.show()


## View Repeatability vs time distance

### Plot 2D histo of signed PWV Repeatability vs Time

In [ ]:
xmin=0
xmax=5
ymin=-1
ymax=1
nbins = 100  # nombre de bins pour les deux axes

fig, axs = plt.subplots(2, 2, figsize=(16, 16))
list_axes = axs.flatten()
N = len(list_axes)
xval = ["dt_repeatinterp_ram_min", "dt_repeatinterp_rum_min",
        "dt_repeatdiff_ram_min",   "dt_repeatdiff_rum_min"]
yval = ["PWV [mm]_ram_repeatinterp_ram", "PWV [mm]_rum_repeatinterp_rum",
        "PWV [mm]_ram_repeatdiff_ram",   "PWV [mm]_rum_repeatdiff_rum"]
titles = ["spectrogram (ram) — interp repeatability",
          "spectrum (rum)    — interp repeatability",
          "spectrogram (ram) — diff repeatability",
          "spectrum (rum)    — diff repeatability"]

for i in range(N):
    ax = list_axes[i]

    # Données propres (NaN exclus, dans les bornes x)
    mask = (
        df_keep[xval[i]].notna() & df_keep[yval[i]].notna()
        & (df_keep[xval[i]] >= xmin) & (df_keep[xval[i]] <= xmax)
    )
    x = df_keep.loc[mask, xval[i]].values
    y = df_keep.loc[mask, yval[i]].values

    # Histogramme 2D avec échelle log pour révéler la concentration vers (0,0)
    h, xedges, yedges, img = ax.hist2d(
        x, y,
        bins=[np.linspace(xmin, xmax, nbins + 1),
              np.linspace(ymin, ymax, nbins + 1)],
        norm=LogNorm(vmin=1),
        cmap="Blues",
    )

    # Colorbar attachée à chaque sous-panneau
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.05)
    fig.colorbar(img, cax=cax, label="counts")

    # Ligne y=0 pour repère visuel
    ax.axhline(0, color="red", lw=0.8, ls="--", alpha=0.7)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel(f"$\\Delta t$ (min)  [{xval[i]}]")
    ax.set_ylabel(f"$\\Delta$ PWV (mm)  [{yval[i]}]")
    ax.set_title(titles[i])

fig.suptitle(f"{the_suptitle}\nRepeatability vs $\\Delta t$ — 2D histogram (log scale)", fontsize=14)
plt.tight_layout()
figname = f"{pathfigs}/{prefix}_{version_run}_hist2d_repeatability_vs_dt_cut{cutstype_name}{figtype}"
plt.savefig(figname)
plt.show()

### Plot 2D histo of absolue PWV Repeatability vs Time

In [ ]:
xmin=0
xmax=5
ymin=0
ymax=1
nbins = 100  # nombre de bins pour les deux axes

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
list_axes = axs.flatten()
N = len(list_axes)
xval = ["dt_repeatinterp_ram_min", "dt_repeatinterp_rum_min",
        "dt_repeatdiff_ram_min",   "dt_repeatdiff_rum_min"]
yval = ["PWV [mm]_ram_repeatinterp_ram", "PWV [mm]_rum_repeatinterp_rum",
        "PWV [mm]_ram_repeatdiff_ram",   "PWV [mm]_rum_repeatdiff_rum"]
titles = ["spectrogram (ram) — interp repeatability",
          "spectrum (rum)    — interp repeatability",
          "spectrogram (ram) — diff repeatability",
          "spectrum (rum)    — diff repeatability"]

for i in range(N):
    ax = list_axes[i]

    # Données propres (NaN exclus, dans les bornes x)
    mask = (
        df_keep[xval[i]].notna() & df_keep[yval[i]].notna()
        & (df_keep[xval[i]] >= xmin) & (df_keep[xval[i]] <= xmax)
    )
    x = df_keep.loc[mask, xval[i]].values
    y = np.abs(df_keep.loc[mask, yval[i]].values)

    # Histogramme 2D avec échelle log pour révéler la concentration vers (0,0)
    h, xedges, yedges, img = ax.hist2d(
        x, y,
        bins=[np.linspace(xmin, xmax, nbins + 1),
              np.linspace(ymin, ymax, nbins + 1)],
        norm=LogNorm(vmin=1),
        cmap="Blues",
    )

    # Colorbar attachée à chaque sous-panneau
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.05)
    fig.colorbar(img, cax=cax, label="counts")

    # Ligne y=0 pour repère visuel
    ax.axhline(0, color="red", lw=0.8, ls="--", alpha=0.7)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel(f"$\\Delta t$ (min)  [{xval[i]}]")
    ax.set_ylabel(f"$\\Delta$ PWV (mm)  [{yval[i]}]")
    ax.set_title(titles[i])

fig.suptitle(f"{the_suptitle}\nRepeatability vs $\\Delta t$ — 2D histogram (log scale)", fontsize=14)
plt.tight_layout()
figname = f"{pathfigs}/{prefix}_{version_run}_hist2d_repeatability_vs_dt_cut{cutstype_name}{figtype}"
plt.savefig(figname)
plt.show()

### Plot profile histogram of absolue PWV Repeatability vs Time, errors beeing std variations

In [ ]:
xmin = 0
xmax = 5
ymin = [ 0, 0, 0, 0 ]
ymax = [ 0.5, 0.5, 1., 1. ]
nbins_profile = 20  # nombre de bins en Δt pour le profil

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
list_axes = axs.flatten()
N = len(list_axes)
xval = ["dt_repeatinterp_ram_min", "dt_repeatinterp_rum_min",
        "dt_repeatdiff_ram_min",   "dt_repeatdiff_rum_min"]
yval = ["PWV [mm]_ram_repeatinterp_ram", "PWV [mm]_rum_repeatinterp_rum",
        "PWV [mm]_ram_repeatdiff_ram",   "PWV [mm]_rum_repeatdiff_rum"]
titles = ["spectrogram (ram) — interp repeatability",
          "spectrum (rum)    — interp repeatability",
          "spectrogram (ram) — diff repeatability",
          "spectrum (rum)    — diff repeatability"]

# Bords et centres des bins en Δt
# Discrétisation de l'axe x (Δt)
bin_edges   = np.linspace(xmin, xmax, nbins_profile + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# loop on plots
for i in range(N):
    ax = list_axes[i]

    # Données propres (NaN exclus, dans les bornes x)
    mask = (
        df_keep[xval[i]].notna() & df_keep[yval[i]].notna()
        & (df_keep[xval[i]] >= xmin) & (df_keep[xval[i]] <= xmax)
    )
    x = df_keep.loc[mask, xval[i]].values
    y = np.abs(df_keep.loc[mask, yval[i]].values)

    # Calcul du profil médiane ± std par bin en x
    # Affectation de chaque point à son bin
    bin_idx = np.clip(np.digitize(x, bin_edges) - 1, 0, nbins_profile - 1)

    # init container before filling
    y_median = np.full(nbins_profile, np.nan)
    y_std    = np.full(nbins_profile, np.nan)
    y_count  = np.zeros(nbins_profile, dtype=int)

    # compute the statistics per x bins
    #Pour chaque bin b on extrait le sous-ensemble sel des valeurs |ΔPWV| correspondantes, 
    #puis :
    #médiane : estimateur robuste du centre de la distribution, insensible aux queues larges — préférable à la moyenne ici car la distribution de |ΔPWV| est asymétrique (semi-définie positive, avec potentiellement des outliers)
    #std (ddof=1) : dispersion des valeurs dans le bin autour de leur propre moyenne — 
    # mesure la largeur de la distribution de répétabilité dans cet intervalle de temps
    #le seuil >= 3 évite d'afficher des barres d'erreur non-significatives sur des bins 
    # quasi-vides
    for b in range(nbins_profile):
        sel = y[bin_idx == b]
        if len(sel) >= 3:  # au moins 3 points pour une statistique fiable
            y_median[b] = np.median(sel)
            y_std[b]    = np.std(sel, ddof=1)
            y_count[b]  = len(sel)

    # Ne tracer que les bins peuplés
    valid = np.isfinite(y_median)

    ax.errorbar(
        bin_centers[valid],
        y_median[valid],
        yerr=y_std[valid],
        fmt="o-",
        color="blue",
        ecolor="blue",
        capsize=4,
        elinewidth=1.2,
        markersize=6,
        label="median ± std",
    )

    # Annotation du nombre de points par bin
    for b in np.where(valid)[0]:
        ax.annotate(
            str(y_count[b]),
            xy=(bin_centers[b], y_median[b] + y_std[b]),
            xytext=(0, 4), textcoords="offset points",
            ha="center", fontsize=10, color="black",
        )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin[i], ymax[i])
    ax.set_xlabel(f"$\\Delta t$ (min)  [{xval[i]}]")
    ax.set_ylabel(r"median $|\Delta\mathrm{PWV}|$ (mm)")
    ax.set_title(titles[i])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f"{the_suptitle}\nMedian $|\\Delta$PWV$|$ profile vs $\\Delta t$ — errorbar = std",
    fontsize=14,
)
plt.tight_layout()
figname = f"{pathfigs}/{prefix}_{version_run}_profile_absPWV_vs_dt_stderr_cut{cutstype_name}{figtype}"
plt.savefig(figname)
plt.show()

### Plot profile histogram of absolue PWV Repeatability vs Time, errors beeing error on mean

In [ ]:
xmin = 0
xmax = 5
ymin = [ 0, 0, 0, 0 ]
ymax = [ 0.5, 0.5, 1., 1. ]
nbins_profile = 20  # nombre de bins en Δt pour le profil

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
list_axes = axs.flatten()
N = len(list_axes)
xval = ["dt_repeatinterp_ram_min", "dt_repeatinterp_rum_min",
        "dt_repeatdiff_ram_min",   "dt_repeatdiff_rum_min"]
yval = ["PWV [mm]_ram_repeatinterp_ram", "PWV [mm]_rum_repeatinterp_rum",
        "PWV [mm]_ram_repeatdiff_ram",   "PWV [mm]_rum_repeatdiff_rum"]
titles = ["spectrogram (ram) — interp repeatability",
          "spectrum (rum)    — interp repeatability",
          "spectrogram (ram) — diff repeatability",
          "spectrum (rum)    — diff repeatability"]

# Bords et centres des bins en Δt
# Discrétisation de l'axe x (Δt)
bin_edges   = np.linspace(xmin, xmax, nbins_profile + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# loop on plots
for i in range(N):
    ax = list_axes[i]

    # Données propres (NaN exclus, dans les bornes x)
    mask = (
        df_keep[xval[i]].notna() & df_keep[yval[i]].notna()
        & (df_keep[xval[i]] >= xmin) & (df_keep[xval[i]] <= xmax)
    )
    x = df_keep.loc[mask, xval[i]].values
    y = np.abs(df_keep.loc[mask, yval[i]].values)

    # Calcul du profil médiane ± std par bin en x
    # Affectation de chaque point à son bin
    bin_idx = np.clip(np.digitize(x, bin_edges) - 1, 0, nbins_profile - 1)

    # init container before filling
    y_median = np.full(nbins_profile, np.nan)
    y_std    = np.full(nbins_profile, np.nan)
    y_count  = np.zeros(nbins_profile, dtype=int)
    y_err    = np.full(nbins_profile, np.nan)
    
    # compute the statistics per x bins
    #Pour chaque bin b on extrait le sous-ensemble sel des valeurs |ΔPWV| correspondantes, 
    #puis :
    #médiane : estimateur robuste du centre de la distribution, insensible aux queues larges — préférable à la moyenne ici car la distribution de |ΔPWV| est asymétrique (semi-définie positive, avec potentiellement des outliers)
    #std (ddof=1) : dispersion des valeurs dans le bin autour de leur propre moyenne — 
    # mesure la largeur de la distribution de répétabilité dans cet intervalle de temps
    #le seuil >= 3 évite d'afficher des barres d'erreur non-significatives sur des bins 
    # quasi-vides
    for b in range(nbins_profile):
        sel = y[bin_idx == b]
        if len(sel) >= 3:  # au moins 3 points pour une statistique fiable
            y_median[b] = np.median(sel)
            y_std[b]    = np.std(sel, ddof=1)
            y_count[b]  = len(sel)
            y_err[b]    = y_std[b]/np.sqrt(y_count[b])
           

    # Ne tracer que les bins peuplés
    valid = np.isfinite(y_median)

    ax.errorbar(
        bin_centers[valid],
        y_median[valid],
        yerr=y_err[valid],
        fmt="o-",
        color="blue",
        ecolor="blue",
        capsize=4,
        elinewidth=1.2,
        markersize=6,
        label="median ± std/sqrt(N)",
    )

    # Annotation du nombre de points par bin
    for b in np.where(valid)[0]:
        ax.annotate(
            str(y_count[b]),
            xy=(bin_centers[b], y_median[b] + y_err[b]),
            xytext=(0, 4), textcoords="offset points",
            ha="center", fontsize=10, color="black",
        )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin[i], ymax[i])
    ax.set_xlabel(f"$\\Delta t$ (min)  [{xval[i]}]")
    ax.set_ylabel(r"median $|\Delta\mathrm{PWV}|$ (mm)")
    ax.set_title(titles[i])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f"{the_suptitle}\nMedian $|\\Delta$PWV$|$ profile vs $\\Delta t$ — errorbar = std",
    fontsize=14,
)
plt.tight_layout()
figname = f"{pathfigs}/{prefix}_{version_run}_profile_absPWV_vs_dt_stderrdivsqN_cut{cutstype_name}{figtype}"
plt.savefig(figname)
plt.show()

### Plot profile histogram of absolue PWV Repeatability vs Time, errors beeing std variations

In [ ]:
xmin = 0
xmax = 5
ymin = [ 0, 0, 0, 0 ]
ymax = [ 0.5, 0.5, 1., 1. ]
nbins_profile = 20  # nombre de bins en Δt pour le profil

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
list_axes = axs.flatten()
N = len(list_axes)
xval = ["dt_repeatinterp_ram_min", "dt_repeatinterp_rum_min",
        "dt_repeatdiff_ram_min",   "dt_repeatdiff_rum_min"]
yval = ["PWV [mm]_ram_repeatinterp_ram", "PWV [mm]_rum_repeatinterp_rum",
        "PWV [mm]_ram_repeatdiff_ram",   "PWV [mm]_rum_repeatdiff_rum"]
titles = ["spectrogram (ram) — interp repeatability",
          "spectrum (rum)    — interp repeatability",
          "spectrogram (ram) — diff repeatability",
          "spectrum (rum)    — diff repeatability"]

# Bords et centres des bins en Δt
# Discrétisation de l'axe x (Δt)
bin_edges   = np.linspace(xmin, xmax, nbins_profile + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# loop on plots
for i in range(N):
    ax = list_axes[i]

    # Données propres (NaN exclus, dans les bornes x)
    mask = (
        df_keep[xval[i]].notna() & df_keep[yval[i]].notna()
        & (df_keep[xval[i]] >= xmin) & (df_keep[xval[i]] <= xmax)
    )
    x = df_keep.loc[mask, xval[i]].values
    y = np.abs(df_keep.loc[mask, yval[i]].values)

    # Calcul du profil médiane ± std par bin en x
    # Affectation de chaque point à son bin
    bin_idx = np.clip(np.digitize(x, bin_edges) - 1, 0, nbins_profile - 1)

    # init container before filling
    y_median = np.full(nbins_profile, np.nan)
    dy2    = np.full(nbins_profile, np.nan)
    ys    = np.full(nbins_profile, np.nan)
    y_count  = np.zeros(nbins_profile, dtype=int)
    ys_err = np.full(nbins_profile, np.nan)

    # compute the statistics per x bins
    #Pour chaque bin b on extrait le sous-ensemble sel des valeurs |ΔPWV| correspondantes, 
    #puis :
    #médiane : estimateur robuste du centre de la distribution, insensible aux queues larges — préférable à la moyenne ici car la distribution de |ΔPWV| est asymétrique (semi-définie positive, avec potentiellement des outliers)
    #std (ddof=1) : dispersion des valeurs dans le bin autour de leur propre moyenne — 
    # mesure la largeur de la distribution de répétabilité dans cet intervalle de temps
    #le seuil >= 3 évite d'afficher des barres d'erreur non-significatives sur des bins 
    # quasi-vides
    for b in range(nbins_profile):
        sel = y[bin_idx == b]
        if len(sel) >= 3:  # au moins 3 points pour une statistique fiable
            y_median[b] = np.median(sel)
            dy2[b]    = np.mean( (sel-y_median[b])**2) # mean of squared deviations wrt mean
            y_count[b]  = len(sel)
            #compute s statistics
            ys[b] =  np.sqrt(dy2[b]*y_count[b]/(y_count[b]-1))
            #compute s spread statistics
            ys_err[b] = ys[b]/np.sqrt(2*(y_count[b]-1))*np.sqrt(y_count[b]) 
           
                                
           
    # Ne tracer que les bins peuplés
    valid = np.isfinite(y_median)

    ax.errorbar(
        bin_centers[valid],
        ys[valid],
        yerr=ys_err[valid],
        fmt="o-",
        color="blue",
        ecolor="blue",
        capsize=4,
        elinewidth=1.2,
        markersize=6,
        label="s ± s_err",
    )

    # Annotation du nombre de points par bin
    for b in np.where(valid)[0]:
        ax.annotate(
            str(y_count[b]),
            xy=(bin_centers[b], ys[b] + ys_err[b]),
            xytext=(0, 4), textcoords="offset points",
            ha="center", fontsize=10, color="black",
        )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin[i], ymax[i])
    ax.set_xlabel(f"$\\Delta t$ (min)  [{xval[i]}]")
    ax.set_ylabel(r" $\sigma (\Delta\mathrm{PWV})$ (mm)")
    ax.set_title(titles[i])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f"{the_suptitle}\nMedian $|\\Delta$PWV$|$ profile vs $\\Delta t$ — errorbar = std",
    fontsize=14,
)
plt.tight_layout()
figname = f"{pathfigs}/{prefix}_{version_run}_profile_absPWV_vs_dt_sigmaerr_cut{cutstype_name}{figtype}"
plt.savefig(figname)
plt.show()

## Fits on PWV Repeatability Profile Histograms

Two fit models are applied to the profile histograms built above:

1. **`mean(|ΔPWV|)` vs Δt** – linear fit `y = a·t + b` restricted to `[0, 3]` min  
   → probes the linear (systematic) growth of the mean absolute PWV difference.

2. **`σ(ΔPWV)` vs Δt** – power-law fit `y = A·t^α`  
   → α ≈ 0.5 would indicate pure Brownian/random-walk behaviour of atmospheric PWV.

Both error conventions (std and std/√N) are handled in separate figures.  
Each panel is annotated with the best-fit parameters and their 1-σ uncertainties
from `scipy.optimize.curve_fit` (weighted least squares, `absolute_sigma=True`).


In [ ]:
# ── Fit imports (scipy) ───────────────────────────────────────────────────────
from scipy.optimize import curve_fit


### Fit helper functions


In [ ]:
# ── Fit model definitions ─────────────────────────────────────────────────────

def linear_model(t, a, b):
    """Linear model:   y = a*t + b"""
    return a * t + b


def power_law_model(t, A, alpha):
    """Power-law model: y = A * t^alpha  (random-walk: alpha ~ 0.5)"""
    return A * np.power(t, alpha)


# ── Profile statistics per Δt bin ─────────────────────────────────────────────

def compute_profile(x, y, bin_edges, min_count=3):
    """
    Compute per-bin statistics of |y|.

    Returns
    -------
    bin_centers, y_median, y_mean, y_std, y_sem, ys, ys_err, y_count
    """
    nbins = len(bin_edges) - 1
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_idx = np.clip(np.digitize(x, bin_edges) - 1, 0, nbins - 1)

    y_median = np.full(nbins, np.nan)
    y_mean   = np.full(nbins, np.nan)
    y_std    = np.full(nbins, np.nan)
    y_sem    = np.full(nbins, np.nan)
    ys       = np.full(nbins, np.nan)
    ys_err   = np.full(nbins, np.nan)
    y_count  = np.zeros(nbins, dtype=int)

    for b in range(nbins):
        sel = y[bin_idx == b]
        n = len(sel)
        if n >= min_count:
            med   = np.median(sel)
            mn    = np.mean(sel)
            std   = np.std(sel, ddof=1)
            dy2   = np.mean((sel - med) ** 2)       # squared deviations wrt median
            s_val = np.sqrt(dy2 * n / (n - 1))      # unbiased spread estimator
            s_err = s_val / np.sqrt(2 * (n - 1)) * np.sqrt(n)  # uncertainty on s

            y_median[b] = med
            y_mean[b]   = mn
            y_std[b]    = std
            y_sem[b]    = std / np.sqrt(n)
            ys[b]       = s_val
            ys_err[b]   = s_err
            y_count[b]  = n

    return bin_centers, y_median, y_mean, y_std, y_sem, ys, ys_err, y_count


# ── Weighted linear fit ───────────────────────────────────────────────────────

def fit_linear(t_vals, y_vals, y_err, t_fit_min=0.0, t_fit_max=3.0):
    """Fit y = a·t + b over [t_fit_min, t_fit_max]."""
    mask = (
        np.isfinite(t_vals) & np.isfinite(y_vals) & np.isfinite(y_err)
        & (y_err > 0)
        & (t_vals >= t_fit_min) & (t_vals <= t_fit_max)
    )
    if mask.sum() < 2:
        return None, None
    try:
        popt, pcov = curve_fit(
            linear_model, t_vals[mask], y_vals[mask],
            sigma=y_err[mask], absolute_sigma=True, p0=[0.05, 0.05],
        )
        return popt, np.sqrt(np.diag(pcov))
    except RuntimeError:
        return None, None


# ── Weighted power-law fit ────────────────────────────────────────────────────

def fit_power_law(t_vals, y_vals, y_err):
    """Fit y = A·t^alpha over all valid (t>0, y>0) points."""
    mask = (
        np.isfinite(t_vals) & np.isfinite(y_vals) & np.isfinite(y_err)
        & (y_err > 0) & (t_vals > 0) & (y_vals > 0)
    )
    if mask.sum() < 2:
        return None, None
    try:
        popt, pcov = curve_fit(
            power_law_model, t_vals[mask], y_vals[mask],
            sigma=y_err[mask], absolute_sigma=True,
            p0=[0.1, 0.5], bounds=([0, 0], [np.inf, 2]),
        )
        return popt, np.sqrt(np.diag(pcov))
    except RuntimeError:
        return None, None


### Shared configuration for fit plots


In [ ]:
# ── Shared configuration ──────────────────────────────────────────────────────

xmin_fit, xmax_fit  = 0, 5
nbins_profile_fit   = 20
tfitmin, tfitmax    = 0.0, 3.0        # linear fit range [min]

bin_edges_fit   = np.linspace(xmin_fit, xmax_fit, nbins_profile_fit + 1)
t_dense         = np.linspace(0.01, xmax_fit, 300)   # dense grid for overlay curves

xval = [
    "dt_repeatinterp_ram_min", "dt_repeatinterp_rum_min",
    "dt_repeatdiff_ram_min",   "dt_repeatdiff_rum_min",
]
yval = [
    "PWV [mm]_ram_repeatinterp_ram", "PWV [mm]_rum_repeatinterp_rum",
    "PWV [mm]_ram_repeatdiff_ram",   "PWV [mm]_rum_repeatdiff_rum",
]
panel_titles = [
    "spectrogram (ram) — interp repeatability",
    "spectrum (rum)    — interp repeatability",
    "spectrogram (ram) — diff repeatability",
    "spectrum (rum)    — diff repeatability",
]
ymax_mean  = [0.5, 0.5, 1.0, 1.0]
ymax_sigma = [0.5, 0.5, 1.0, 1.0]

props_box = dict(boxstyle="round", facecolor="wheat", alpha=0.85)


### Fig A & B — Linear fit on mean(|ΔPWV|) vs Δt [0–3 min]


In [ ]:
# ── Figure A: mean(|ΔPWV|) vs Δt — linear fit — errorbar = std ───────────────

fig_A, axs_A = plt.subplots(2, 2, figsize=(16, 12))

for i, ax in enumerate(axs_A.flatten()):
    mask = (
        df_keep[xval[i]].notna() & df_keep[yval[i]].notna()
        & (df_keep[xval[i]] >= xmin_fit) & (df_keep[xval[i]] <= xmax_fit)
    )
    x = df_keep.loc[mask, xval[i]].values
    y = np.abs(df_keep.loc[mask, yval[i]].values)

    (bc, y_med, y_mn, y_std, y_sem,
     ys, ys_err, y_cnt) = compute_profile(x, y, bin_edges_fit)
    valid = np.isfinite(y_mn)

    # Data points
    ax.errorbar(bc[valid], y_mn[valid], yerr=y_std[valid],
                fmt="o-", color="blue", ecolor="steelblue",
                capsize=4, elinewidth=1.2, markersize=6,
                label=r"mean $|\Delta\mathrm{PWV}|$ ± std")

    for b in np.where(valid)[0]:
        ax.annotate(str(y_cnt[b]),
                    xy=(bc[b], y_mn[b] + y_std[b]),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=8, color="dimgray")

    # Linear fit
    popt, perr = fit_linear(bc[valid], y_mn[valid], y_std[valid],
                             tfitmin, tfitmax)
    if popt is not None:
        t_fit_plot = np.linspace(tfitmin, tfitmax, 200)
        ax.plot(t_fit_plot, linear_model(t_fit_plot, *popt),
                color="red", lw=2, ls="--",
                label=f"Linear fit [{tfitmin}–{tfitmax} min]")
        txt = (f"Linear fit: $y = a\,t + b$\n"
               f"$a = {popt[0]:.3f} \\pm {perr[0]:.3f}$ mm/min\n"
               f"$b = {popt[1]:.3f} \\pm {perr[1]:.3f}$ mm")
        ax.text(0.97, 0.05, txt, transform=ax.transAxes,
                fontsize=12, va="bottom", ha="right", bbox=props_box)
    else:
        ax.text(0.97, 0.05, "Fit failed", transform=ax.transAxes,
                fontsize=12, va="bottom", ha="right", color="red")

    ax.set_xlim(xmin_fit, xmax_fit); ax.set_ylim(0, ymax_mean[i])
    ax.set_xlabel(r"$\Delta t$ (min)")
    ax.set_ylabel(r"mean $|\Delta\mathrm{PWV}|$ (mm)")
    ax.set_title(panel_titles[i]); ax.legend(fontsize=12,loc="upper left"); ax.grid(True, alpha=0.3)

fig_A.suptitle(
    f"{the_suptitle}\n"
    r"mean $|\Delta\mathrm{PWV}|$ vs $\Delta t$ — linear fit [0–3 min] — err = std",
    fontsize=13)
plt.tight_layout()
fn = f"{pathfigs}/{prefix}_{version_run}_fit_linear_meanAbsPWV_std_cut{cutstype_name}{figtype}"
plt.savefig(fn); plt.show(); print(f"Saved: {fn}")


In [ ]:
# ── Figure B: mean(|ΔPWV|) vs Δt — linear fit — errorbar = std/√N ───────────

fig_B, axs_B = plt.subplots(2, 2, figsize=(16, 12))

for i, ax in enumerate(axs_B.flatten()):
    mask = (
        df_keep[xval[i]].notna() & df_keep[yval[i]].notna()
        & (df_keep[xval[i]] >= xmin_fit) & (df_keep[xval[i]] <= xmax_fit)
    )
    x = df_keep.loc[mask, xval[i]].values
    y = np.abs(df_keep.loc[mask, yval[i]].values)

    (bc, y_med, y_mn, y_std, y_sem,
     ys, ys_err, y_cnt) = compute_profile(x, y, bin_edges_fit)
    valid = np.isfinite(y_mn)

    ax.errorbar(bc[valid], y_mn[valid], yerr=y_sem[valid],
                fmt="o-", color="blue", ecolor="steelblue",
                capsize=4, elinewidth=1.2, markersize=6,
                label=r"mean $|\Delta\mathrm{PWV}|$ ± std/$\sqrt{N}$")

    for b in np.where(valid)[0]:
        ax.annotate(str(y_cnt[b]),
                    xy=(bc[b], y_mn[b] + y_sem[b]),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=8, color="dimgray")

    popt, perr = fit_linear(bc[valid], y_mn[valid], y_sem[valid],
                             tfitmin, tfitmax)
    if popt is not None:
        t_fit_plot = np.linspace(tfitmin, tfitmax, 200)
        ax.plot(t_fit_plot, linear_model(t_fit_plot, *popt),
                color="red", lw=2, ls="--",
                label=f"Linear fit [{tfitmin}–{tfitmax} min]")
        txt = (f"Linear fit: $y = a\,t + b$\n"
               f"$a = {popt[0]:.3f} \\pm {perr[0]:.3f}$ mm/min\n"
               f"$b = {popt[1]:.3f} \\pm {perr[1]:.3f}$ mm")
        ax.text(0.97, 0.05, txt, transform=ax.transAxes,
                fontsize=12, va="bottom", ha="right", bbox=props_box)
    else:
        ax.text(0.97, 0.05, "Fit failed", transform=ax.transAxes,
                fontsize=12, va="bottom", ha="right", color="red")

    ax.set_xlim(xmin_fit, xmax_fit); ax.set_ylim(0, ymax_mean[i])
    ax.set_xlabel(r"$\Delta t$ (min)")
    ax.set_ylabel(r"mean $|\Delta\mathrm{PWV}|$ (mm)")
    ax.set_title(panel_titles[i]); ax.legend(fontsize=12,loc="upper left"); ax.grid(True, alpha=0.3)

fig_B.suptitle(
    f"{the_suptitle}\n"
    r"mean $|\Delta\mathrm{PWV}|$ vs $\Delta t$ — linear fit [0–3 min] — err = std/$\sqrt{N}$",
    fontsize=13)
plt.tight_layout()
fn = f"{pathfigs}/{prefix}_{version_run}_fit_linear_meanAbsPWV_sem_cut{cutstype_name}{figtype}"
plt.savefig(fn); plt.show(); print(f"Saved: {fn}")


### Fig C & D — Power-law fit on σ(ΔPWV) vs Δt (random-walk signature)


In [ ]:
# ── Figure C: σ(ΔPWV) vs Δt — power-law fit — errorbar = s_err (spread) ─────

fig_C, axs_C = plt.subplots(2, 2, figsize=(16, 12))

for i, ax in enumerate(axs_C.flatten()):
    mask = (
        df_keep[xval[i]].notna() & df_keep[yval[i]].notna()
        & (df_keep[xval[i]] >= xmin_fit) & (df_keep[xval[i]] <= xmax_fit)
    )
    x = df_keep.loc[mask, xval[i]].values
    y = np.abs(df_keep.loc[mask, yval[i]].values)

    (bc, y_med, y_mn, y_std, y_sem,
     ys, ys_err, y_cnt) = compute_profile(x, y, bin_edges_fit)
    valid = np.isfinite(ys)

    ax.errorbar(bc[valid], ys[valid], yerr=ys_err[valid],
                fmt="s-", color="blue", ecolor="steelblue",
                capsize=4, elinewidth=1.2, markersize=6,
                label=r"$\sigma(\Delta\mathrm{PWV})$ = $s$ ± $s_{\rm err}$")

    for b in np.where(valid)[0]:
        ax.annotate(str(y_cnt[b]),
                    xy=(bc[b], ys[b] + ys_err[b]),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=8, color="dimgray")

    popt, perr = fit_power_law(bc[valid], ys[valid], ys_err[valid])
    if popt is not None:
        A, alpha = popt; dA, dalpha = perr
        ax.plot(t_dense, power_law_model(t_dense, A, alpha),
                color="crimson", lw=2, ls="--", label=r"$A\,t^{\alpha}$ fit")
        # Reference pure random-walk curve (rescaled to first data point)
        A_rw = ys[valid][0] / (bc[valid][0] ** 0.5) if bc[valid][0] > 0 else A
        ax.plot(t_dense, A_rw * np.sqrt(t_dense),
                color="gray", lw=1.2, ls=":", label=r"$\propto \sqrt{t}$ (ref.)")
        txt = (r"Power-law: $y = A\,t^{\alpha}$" + "\n"
               f"$A = {A:.3f} \\pm {dA:.3f}$ mm\n"
               f"$\\alpha = {alpha:.3f} \\pm {dalpha:.3f}$")
        ax.text(0.03, 0.95, txt, transform=ax.transAxes,
                fontsize=12, va="top", ha="left", bbox=props_box)
    else:
        ax.text(0.03, 0.95, "Fit failed", transform=ax.transAxes,
                fontsize=12, va="top", ha="left", color="red")

    ax.set_xlim(xmin_fit, xmax_fit); ax.set_ylim(0, ymax_sigma[i])
    ax.set_xlabel(r"$\Delta t$ (min)")
    ax.set_ylabel(r"$\sigma(\Delta\mathrm{PWV})$ (mm)")
    ax.set_title(panel_titles[i]); ax.legend(fontsize=12, loc="upper right"); ax.grid(True, alpha=0.3)

fig_C.suptitle(
    f"{the_suptitle}\n"
    r"$\sigma(\Delta\mathrm{PWV})$ vs $\Delta t$ — power-law $A\,t^{\alpha}$ — err = $s_{\rm err}$",
    fontsize=13)
plt.tight_layout()
fn = f"{pathfigs}/{prefix}_{version_run}_fit_powerlaw_sigmaPWV_serr_cut{cutstype_name}{figtype}"
plt.savefig(fn); plt.show(); print(f"Saved: {fn}")


In [ ]:
# ── Figure D: σ(ΔPWV) vs Δt — power-law fit — errorbar = std/√N ─────────────

fig_D, axs_D = plt.subplots(2, 2, figsize=(16, 12))

for i, ax in enumerate(axs_D.flatten()):
    mask = (
        df_keep[xval[i]].notna() & df_keep[yval[i]].notna()
        & (df_keep[xval[i]] >= xmin_fit) & (df_keep[xval[i]] <= xmax_fit)
    )
    x = df_keep.loc[mask, xval[i]].values
    y = np.abs(df_keep.loc[mask, yval[i]].values)

    (bc, y_med, y_mn, y_std, y_sem,
     ys, ys_err, y_cnt) = compute_profile(x, y, bin_edges_fit)
    valid = np.isfinite(y_std)

    ax.errorbar(bc[valid], y_std[valid], yerr=y_sem[valid],
                fmt="s-", color="blue", ecolor="steelblue",
                capsize=4, elinewidth=1.2, markersize=6,
                label=r"$\sigma(\Delta\mathrm{PWV})$ = std ± std/$\sqrt{N}$")

    for b in np.where(valid)[0]:
        ax.annotate(str(y_cnt[b]),
                    xy=(bc[b], y_std[b] + y_sem[b]),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=8, color="dimgray")

    popt, perr = fit_power_law(bc[valid], y_std[valid], y_sem[valid])
    if popt is not None:
        A, alpha = popt; dA, dalpha = perr
        ax.plot(t_dense, power_law_model(t_dense, A, alpha),
                color="crimson", lw=2, ls="--", label=r"$A\,t^{\alpha}$ fit")
        A_rw = y_std[valid][0] / (bc[valid][0] ** 0.5) if bc[valid][0] > 0 else A
        ax.plot(t_dense, A_rw * np.sqrt(t_dense),
                color="gray", lw=1.2, ls=":", label=r"$\propto \sqrt{t}$ (ref.)")
        txt = (r"Power-law: $y = A\,t^{\alpha}$" + "\n"
               f"$A = {A:.3f} \\pm {dA:.3f}$ mm\n"
               f"$\\alpha = {alpha:.3f} \\pm {dalpha:.3f}$")
        ax.text(0.03, 0.95, txt, transform=ax.transAxes,
                fontsize=12, va="top", ha="left", bbox=props_box)
    else:
        ax.text(0.03, 0.95, "Fit failed", transform=ax.transAxes,
                fontsize=12, va="top", ha="left", color="red")

    ax.set_xlim(xmin_fit, xmax_fit); ax.set_ylim(0, ymax_sigma[i])
    ax.set_xlabel(r"$\Delta t$ (min)")
    ax.set_ylabel(r"$\sigma(\Delta\mathrm{PWV})$ (mm)")
    ax.set_title(panel_titles[i]); ax.legend(fontsize=12, loc="upper right"); ax.grid(True, alpha=0.3)

fig_D.suptitle(
    f"{the_suptitle}\n"
    r"$\sigma(\Delta\mathrm{PWV})$ vs $\Delta t$ — power-law $A\,t^{\alpha}$ — err = std/$\sqrt{N}$",
    fontsize=13)
plt.tight_layout()
fn = f"{pathfigs}/{prefix}_{version_run}_fit_powerlaw_sigmaPWV_sem_cut{cutstype_name}{figtype}"
plt.savefig(fn); plt.show(); print(f"Saved: {fn}")
